In [10]:
import os
import time
import subprocess
import pandas as pd

from pathlib import Path
from PIL import Image

env = os.environ.copy()
env['LD_LIBRARY_PATH'] = '/usr/local/lib:' + env.get('LD_LIBRARY_PATH', '')
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '')

In [11]:
def compressImg(input_path, output_path, codec="gzip", level=6):
    """
    Compress a file using gzip, bzip2, or zstd.

    Args:
        input_path (str): Path to the input file.
        output_path (str): Path to save the compressed file.
        codec (str): Compression codec ('gzip', 'bzip2', 'zstd').
        level (int): Compression level (1=fastest, 9=slowest for gzip/bzip2; 19 for zstd).

    Returns:
        bool: True if compression is successful, False otherwise.
    """
    cmd = []
    if codec == "gzip":
        cmd = ["gzip", "-c", f"-{level}", input_path]
    elif codec == "bzip2":
        cmd = ["bzip2", "-c", f"-{level}", input_path]
    elif codec == "zstd":
        cmd = ["zstd", f"-{level}", "-o", output_path, input_path]
    else:
        raise ValueError(f"Unsupported compression codec: {codec}")

    if codec in ["gzip", "bzip2"]:
        with open(output_path, "wb") as outfile:
            try:
                subprocess.run(cmd, stdout=outfile, check=True)
            except Exception as e:
                print(f"Error compressing {input_path} with {codec}: {e}.")
                return False
    else:
        try:
            subprocess.run(cmd, check=True)
        except Exception as e:
            print(f"Error compressing {input_path} with {codec}: {e}.")
            return False

    return True


def decompressImg(input_path, output_path, codec="gzip"):
    """
    Decompress a file using gzip, bzip2, or zstd.

    Args:
        input_path (str): Path to the compressed file.
        output_path (str): Path to save the decompressed file.
        codec (str): Decompression codec ('gzip', 'bzip2', 'zstd').

    Returns:
        bool: True if decompression is successful, False otherwise.
    """
    cmd = []
    if codec == "gzip":
        cmd = ["gzip", "-dc", input_path]
    elif codec == "bzip2":
        cmd = ["bzip2", "-dc", input_path]
    elif codec == "zstd":
        cmd = ["zstd", "-d", "-o", output_path, input_path]
    else:
        raise ValueError(f"Unsupported decompression codec: {codec}")

    if codec in ["gzip", "bzip2"]:
        with open(output_path, "wb") as outfile:
            try:
                subprocess.run(cmd, stdout=outfile, check=True, env=env)
            except Exception as e:
                print(f"Error decompressing {input_path} with {codec}: {e}.")
                return False
    else:
        try:
            subprocess.run(cmd, check=True, env=env)
        except Exception as e:
            print(f"Error decompressing {input_path} with {codec}: {e}.")
            return False

    return True


In [12]:
def cal_compression_ratio(original_path, compressed_path):
    """
    Calculate compression rate based on file sizes.
    """
    original_size = os.path.getsize(original_path)
    compressed_size = os.path.getsize(compressed_path)

    with Image.open(original_path) as img:
        width, height = img.size
        npixels = width * height
    bpsp = (compressed_size * 8) / (npixels * 3)
    compression_ratio = compressed_size / original_size
    
    return compression_ratio, original_size, compressed_size, bpsp, width, height, npixels

In [13]:
def process_dataset(dataset_path, output_dir, codec, fmt, mode="compress", cal_metric=True, verbose=False):
    """
    Process a dataset (Kodak or CLIC) using JPEG-XL, with optional compression rate calculation.
    """
    dataset_path = Path(dataset_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # imgs = list(dataset_path.glob('*.png'))
    # imgs = list(dataset_path.glob('*.jpg')) + list(dataset_path.glob('*.jpeg')) + list(dataset_path.glob('*.png'))
    imgs = list(filter(lambda x: x.is_file() and x.suffix.lower() in ['.jpg', '.jpeg', '.png'], dataset_path.glob('*')))
    if len(imgs) == 0:
        print(f'No Imgs found in {dataset_path}.')
        return
    
    rows = []
    for img in imgs:
        output_path = output_dir / f'{img.stem}.{fmt}' if mode == 'compress' else output_dir / f'{img.stem}_recon.png'
        
        start = time.time()
        if mode == 'compress':
            success = compressImg(img, output_path, codec=codec)
        elif mode == 'decompress':
            success = decompressImg(img, output_path, codec=codec)
        elapsed = time.time() - start   

        if success and cal_metric and mode == 'compress':
            compression_ratio, original_size, compressed_size, bpsp, width, height, npixels = cal_compression_ratio(img, output_path)
            if verbose:
                print(
                    f'{img.name}: compression ratio = {compression_ratio:.2%} (Original: {original_size} bytes, Compressed: {compressed_size} bytes), bpsp: {bpsp}.'
                )
            rows.append([img.name, bpsp, compression_ratio, original_size, compressed_size, width, height, npixels, elapsed])
            
    return rows

In [14]:
''' 1. Touch and Go '''

from tqdm import tqdm
fmts = {
    'gzip':  'zip',
    'bzip2': 'bz2',
    'zstd':  'zst'
}

for codec in tqdm(['gzip', 'zstd', 'bzip2']):
    # dataset_path = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp/test/image'
    # output_dir   = f'/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/{codec}'

    # rows = process_dataset(
    #     dataset_path, output_dir, codec=codec, fmt=fmts[codec], mode='compress', verbose=False
    # )
    # df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
    # df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
    
    # print(f'Codec: {codec}')
    # # display(df)
    # display(df.describe())
    # print('\n\n')
    # df.to_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_TouchandGoDataset-v2.csv', index=False)

    print(codec)
    display(pd.read_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_TouchandGoDataset-v2.csv').describe())

  0%|          | 0/3 [00:00<?, ?it/s]

gzip


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,2.297492,0.915953,288034.526429,264671.100000,640.0,480.0,307200.0,0.017268,2.500300
std,0.409295,0.028013,44205.507025,47150.821362,0.0,0.0,0.0,0.003123,0.383728
min,1.360035,0.814544,192348.000000,156676.000000,640.0,480.0,307200.0,0.009703,1.669687
25%,1.992784,0.900977,253517.500000,229568.750000,640.0,480.0,307200.0,0.015604,2.200673
50%,2.209562,0.916228,276483.000000,254541.500000,640.0,480.0,307200.0,0.017035,2.400026
75%,2.468735,0.934562,310647.000000,284398.250000,640.0,480.0,307200.0,0.019599,2.696589
max,4.088090,0.978904,481097.000000,470948.000000,640.0,480.0,307200.0,0.028545,4.176189


zstd


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,2.363059,0.942767,288034.526429,272224.417143,640.0,480.0,307200.0,0.011234,2.500300
std,0.408743,0.025137,44205.507025,47087.238534,0.0,0.0,0.0,0.001509,0.383728
min,1.437413,0.855232,192348.000000,165590.000000,640.0,480.0,307200.0,0.007175,1.669687
25%,2.057780,0.928818,253517.500000,237056.250000,640.0,480.0,307200.0,0.010701,2.200673
50%,2.275803,0.944628,276483.000000,262172.500000,640.0,480.0,307200.0,0.011286,2.400026
75%,2.542852,0.960493,310647.000000,292936.500000,640.0,480.0,307200.0,0.012070,2.696589
max,4.176406,1.000057,481097.000000,481122.000000,640.0,480.0,307200.0,0.030847,4.176189


bzip2


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,2.288235,0.912797,288034.526429,263604.626429,640.0,480.0,307200.0,0.037714,2.500300
std,0.400374,0.027719,44205.507025,46123.047401,0.0,0.0,0.0,0.007189,0.383728
min,1.386771,0.826870,192348.000000,159756.000000,640.0,480.0,307200.0,0.022264,1.669687
25%,1.993451,0.894004,253517.500000,229645.500000,640.0,480.0,307200.0,0.031640,2.200673
50%,2.207652,0.912144,276483.000000,254321.500000,640.0,480.0,307200.0,0.036542,2.400026
75%,2.424204,0.932929,310647.000000,279268.250000,640.0,480.0,307200.0,0.042376,2.696589
max,4.038576,0.969613,481097.000000,465244.000000,640.0,480.0,307200.0,0.066129,4.176189


100%|██████████| 3/3 [00:00<00:00, 33.72it/s]


In [15]:
''' 2. ObjectFolder '''

from tqdm import tqdm
fmts = {
    'gzip':  'zip',
    'bzip2': 'bz2',
    'zstd':  'zst'
}

for codec in tqdm(['gzip', 'zstd', 'bzip2']):
    dataset_path = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp/test/image'
    output_dir   = f'/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/{codec}'

    # rows = process_dataset(
    #     dataset_path, output_dir, codec=codec, fmt=fmts[codec], mode='compress', verbose=False
    # )
    # df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
    # df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
    
    # print(f'Codec: {codec}')
    # # display(df)
    # display(df.describe())
    # print('\n\n')
    # df.to_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_ObjectFolder_1.0.csv', index=False)

    print(codec)
    display(pd.read_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_ObjectFolder_1.0.csv').describe())

  0%|          | 0/3 [00:00<?, ?it/s]

gzip


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,3.968917,1.001209,28541.906000,28576.200300,160.0,120.0,19200.0,0.003542,3.964154
std,0.374463,0.000085,2695.165442,2696.134858,0.0,0.0,0.0,0.000585,0.374329
min,3.364028,1.000929,24187.000000,24221.000000,160.0,120.0,19200.0,0.002244,3.359306
25%,3.683889,1.001142,26490.000000,26524.000000,160.0,120.0,19200.0,0.003095,3.679167
50%,3.851667,1.001221,27698.000000,27732.000000,160.0,120.0,19200.0,0.003440,3.846944
75%,4.187778,1.001279,30118.000000,30152.000000,160.0,120.0,19200.0,0.003913,4.183056
max,5.833472,1.001431,41962.000000,42001.000000,160.0,120.0,19200.0,0.011261,5.828056


zstd


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,3.966098,1.000495,28541.906000,28555.906000,160.0,120.0,19200.0,0.003120,3.964154
std,0.374329,0.000043,2695.165442,2695.165442,0.0,0.0,0.0,0.000593,0.374329
min,3.361250,1.000334,24187.000000,24201.000000,160.0,120.0,19200.0,0.002149,3.359306
25%,3.681111,1.000465,26490.000000,26504.000000,160.0,120.0,19200.0,0.002601,3.679167
50%,3.848889,1.000505,27698.000000,27712.000000,160.0,120.0,19200.0,0.003089,3.846944
75%,4.185000,1.000529,30118.000000,30132.000000,160.0,120.0,19200.0,0.003495,4.183056
max,5.830000,1.000579,41962.000000,41976.000000,160.0,120.0,19200.0,0.009958,5.828056


bzip2


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,4.031012,1.016982,28541.906000,29023.289250,160.0,120.0,19200.0,0.007975,3.964154
std,0.375374,0.001270,2695.165442,2702.692357,0.0,0.0,0.0,0.001537,0.374329
min,3.424306,1.012306,24187.000000,24655.000000,160.0,120.0,19200.0,0.005810,3.359306
25%,3.745000,1.016109,26490.000000,26964.000000,160.0,120.0,19200.0,0.006544,3.679167
50%,3.913194,1.017254,27698.000000,28175.000000,160.0,120.0,19200.0,0.007362,3.846944
75%,4.250417,1.017958,30118.000000,30603.000000,160.0,120.0,19200.0,0.009296,4.183056
max,5.900694,1.020080,41962.000000,42485.000000,160.0,120.0,19200.0,0.017804,5.828056


100%|██████████| 3/3 [00:00<00:00, 22.37it/s]


In [16]:
''' 3. SSVTP '''

from tqdm import tqdm
fmts = {
    'gzip':  'zip',
    'bzip2': 'bz2',
    'zstd':  'zst'
}

for codec in tqdm(['gzip', 'zstd', 'bzip2']):
    dataset_path = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp/test'
    output_dir   = f'/data/ssd/zhaoy/datasets/SSVTP/compressed/{codec}'

    # rows = process_dataset(
    #     dataset_path, output_dir, codec=codec, fmt=fmts[codec], mode='compress', verbose=False
    # )
    # df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
    # df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
    
    # print(f'Codec: {codec}')
    # # display(df)
    # display(df.describe())
    # print('\n\n')
    # df.to_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_SSVTP.csv', index=False)

    print(codec)
    display(pd.read_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_SSVTP.csv').describe())

  0%|          | 0/3 [00:00<?, ?it/s]

gzip


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,2.234042,1.000603,64301.678649,64340.404139,240.0,320.0,76800.0,0.004953,2.232697
std,0.168593,0.000025,4853.413152,4855.490066,0.0,0.0,0.0,0.000591,0.168521
min,1.872778,1.000523,53899.000000,53936.000000,240.0,320.0,76800.0,0.003201,1.871493
25%,2.091814,1.000584,60208.000000,60244.250000,240.0,320.0,76800.0,0.004645,2.090556
50%,2.212049,1.000604,63670.500000,63707.000000,240.0,320.0,76800.0,0.004919,2.210781
75%,2.359618,1.000621,67915.000000,67957.000000,240.0,320.0,76800.0,0.005219,2.358160
max,2.688924,1.000686,77399.000000,77441.000000,240.0,320.0,76800.0,0.007656,2.687465


zstd


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,2.233210,1.000230,64301.678649,64316.445534,240.0,320.0,76800.0,0.003918,2.232697
std,0.168550,0.000009,4853.413152,4854.237255,0.0,0.0,0.0,0.000455,0.168521
min,1.871979,1.000207,53899.000000,53913.000000,240.0,320.0,76800.0,0.002591,1.871493
25%,2.091042,1.000223,60208.000000,60222.000000,240.0,320.0,76800.0,0.003752,2.090556
50%,2.211267,1.000231,63670.500000,63684.500000,240.0,320.0,76800.0,0.003878,2.210781
75%,2.358715,1.000237,67915.000000,67931.000000,240.0,320.0,76800.0,0.004041,2.358160
max,2.688021,1.000260,77399.000000,77415.000000,240.0,320.0,76800.0,0.006316,2.687465


bzip2


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,2.254606,1.009840,64301.678649,64932.663399,240.0,320.0,76800.0,0.015337,2.232697
std,0.169364,0.000395,4853.413152,4877.681383,0.0,0.0,0.0,0.001561,0.168521
min,1.891319,1.008650,53899.000000,54470.000000,240.0,320.0,76800.0,0.010127,1.871493
25%,2.111953,1.009535,60208.000000,60824.250000,240.0,320.0,76800.0,0.014663,2.090556
50%,2.232674,1.009843,63670.500000,64301.000000,240.0,320.0,76800.0,0.015505,2.210781
75%,2.380590,1.010139,67915.000000,68561.000000,240.0,320.0,76800.0,0.016322,2.358160
max,2.711667,1.010907,77399.000000,78096.000000,240.0,320.0,76800.0,0.019607,2.687465


100%|██████████| 3/3 [00:00<00:00, 56.40it/s]


In [17]:
''' 4. ObjTac '''

from tqdm import tqdm
fmts = {
    'gzip':  'zip',
    'bzip2': 'bz2',
    'zstd':  'zst'
}

for codec in tqdm(['gzip', 'zstd', 'bzip2']):
    dataset_path = '/data/ssd/zhaoy/datasets/ObjTac/dataset-comp/test/image'
    output_dir   = f'/data/ssd/zhaoy/datasets/ObjTac/compressed/{codec}'

    # rows = process_dataset(
    #     dataset_path, output_dir, codec=codec, fmt=fmts[codec], mode='compress', verbose=False
    # )
    # df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
    # df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
    
    # print(f'Codec: {codec}')
    # # display(df)
    # display(df.describe())
    # print('\n\n')
    # df.to_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_ObjTac.csv', index=False)

    print(codec)
    display(pd.read_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_ObjTac.csv').describe())

  0%|          | 0/3 [00:00<?, ?it/s]

gzip


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000,51.000000
mean,0.570655,0.937335,11784.803922,11616.627451,60.0,903.352941,54201.176471,0.034419,0.578609
std,0.339258,0.131956,7995.848535,8139.666477,0.0,391.917891,23515.073434,0.051689,0.328989
min,0.026573,0.418093,818.000000,342.000000,60.0,331.000000,19860.000000,0.002099,0.063559
25%,0.239417,0.943736,4868.500000,4217.000000,60.0,587.500000,35250.000000,0.003896,0.253969
50%,0.630676,0.997622,10936.000000,10868.000000,60.0,813.000000,48780.000000,0.010940,0.629227
75%,0.818494,1.002764,19075.500000,19008.500000,60.0,1142.000000,68520.000000,0.042868,0.826285
max,1.250434,1.009554,28278.000000,28327.000000,60.0,1802.000000,108120.000000,0.266483,1.246181


zstd


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000,51.000000
mean,0.568127,0.926763,11784.803922,11582.647059,60.0,903.352941,54201.176471,0.027263,0.578609
std,0.340472,0.141716,7995.848535,8167.242340,0.0,391.917891,23515.073434,0.032558,0.328989
min,0.024087,0.378973,818.000000,310.000000,60.0,331.000000,19860.000000,0.003046,0.063559
25%,0.232669,0.934895,4868.500000,4144.000000,60.0,587.500000,35250.000000,0.011867,0.253969
50%,0.629650,1.000558,10936.000000,10833.000000,60.0,813.000000,48780.000000,0.020222,0.629227
75%,0.816032,1.000790,19075.500000,19089.500000,60.0,1142.000000,68520.000000,0.028924,0.826285
max,1.247396,1.002787,28278.000000,28292.000000,60.0,1802.000000,108120.000000,0.211814,1.246181


bzip2


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000,51.000000
mean,0.594279,1.000061,11784.803922,12015.431373,60.0,903.352941,54201.176471,0.049741,0.578609
std,0.342246,0.115022,7995.848535,8173.904693,0.0,391.917891,23515.073434,0.071292,0.328989
min,0.033411,0.525672,818.000000,430.000000,60.0,331.000000,19860.000000,0.002266,0.063559
25%,0.272130,1.014888,4868.500000,4649.000000,60.0,587.500000,35250.000000,0.006150,0.253969
50%,0.681522,1.023813,10936.000000,11318.000000,60.0,813.000000,48780.000000,0.011687,0.629227
75%,0.843330,1.041959,19075.500000,19435.500000,60.0,1142.000000,68520.000000,0.084714,0.826285
max,1.288194,1.182161,28278.000000,28729.000000,60.0,1802.000000,108120.000000,0.307513,1.246181


100%|██████████| 3/3 [00:00<00:00, 64.01it/s]


In [19]:
''' 5. YCB-Slide  '''

from tqdm import tqdm
fmts = {
    'gzip':  'zip',
    'bzip2': 'bz2',
    'zstd':  'zst'
}

for codec in tqdm(['gzip', 'zstd', 'bzip2']):
    dataset_path = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp/test/image'
    output_dir   = f'/data/ssd/zhaoy/datasets/YCB-Slide/compressed/{codec}'

    # rows = process_dataset(
    #     dataset_path, output_dir, codec=codec, fmt=fmts[codec], mode='compress', verbose=False
    # )
    # df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
    # df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
    
    # print(f'Codec: {codec}')
    # # display(df)
    # display(df.describe())
    # print('\n\n')
    # df.to_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_YCB-Slide.csv', index=False)

    print(codec)
    display(pd.read_csv(f'/home/zhaoy/TaCo-Bench/lossless/statistics/{codec}_YCB-Slide.csv').describe())

  0%|          | 0/3 [00:00<?, ?it/s]

gzip


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000,10916.000000
mean,2.185354,1.000896,62881.925247,62938.201356,240.0,320.0,76800.0,0.004372,2.183400
std,0.096873,0.000077,2789.584906,2789.946572,0.0,0.0,0.0,0.000949,0.096861
min,1.885521,1.000735,54243.000000,54303.000000,240.0,320.0,76800.0,0.003068,1.883438
25%,2.117500,1.000833,60928.750000,60984.000000,240.0,320.0,76800.0,0.003572,2.115582
50%,2.177465,1.000888,62656.000000,62711.000000,240.0,320.0,76800.0,0.003942,2.175556
75%,2.243073,1.000963,64545.250000,64600.500000,240.0,320.0,76800.0,0.005050,2.241155
max,2.557083,1.001106,73586.000000,73644.000000,240.0,320.0,76800.0,0.011064,2.555069


zstd


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000,10916.000000
mean,2.183896,1.000227,62881.925247,62896.190546,240.0,320.0,76800.0,0.003511,2.183400
std,0.096877,0.000008,2789.584906,2790.055990,0.0,0.0,0.0,0.000763,0.096861
min,1.883924,1.000213,54243.000000,54257.000000,240.0,320.0,76800.0,0.002489,1.883438
25%,2.116068,1.000220,60928.750000,60942.750000,240.0,320.0,76800.0,0.002858,2.115582
50%,2.176042,1.000226,62656.000000,62670.000000,240.0,320.0,76800.0,0.003270,2.175556
75%,2.241641,1.000233,64545.250000,64559.250000,240.0,320.0,76800.0,0.003974,2.241155
max,2.555625,1.000258,73586.000000,73602.000000,240.0,320.0,76800.0,0.009461,2.555069


bzip2


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000,10916.000000
mean,2.205057,1.009929,62881.925247,63505.630542,240.0,320.0,76800.0,0.013549,2.183400
std,0.097308,0.000287,2789.584906,2802.484020,0.0,0.0,0.0,0.002746,0.096861
min,1.903993,1.008651,54243.000000,54835.000000,240.0,320.0,76800.0,0.009628,1.883438
25%,2.136901,1.009743,60928.750000,61542.750000,240.0,320.0,76800.0,0.010972,2.115582
50%,2.197101,1.009937,62656.000000,63276.500000,240.0,320.0,76800.0,0.012037,2.175556
75%,2.263064,1.010125,64545.250000,65176.250000,240.0,320.0,76800.0,0.016159,2.241155
max,2.578229,1.010957,73586.000000,74253.000000,240.0,320.0,76800.0,0.028102,2.555069


100%|██████████| 3/3 [00:00<00:00, 34.22it/s]
